# 🎓 ProctorAI - Báo Cáo Kỹ Thuật: Phát Hiện Khuôn Mặt & Ước Lượng Hướng Đầu
> **Dự án Capstone:** Samsung Innovation Campus (SIC)  
> **Thành viên:** Hiếu & Rin  
> **Mục tiêu Notebook:** Bóc tách trực quan từng bước thuật toán phát hiện khuôn mặt và cách biến các điểm ảnh 2D thành góc quay đầu 3D (Pitch, Yaw, Roll).

---
### 📌 6 Bước Thực Nghiệm Tinh Gọn:
1. **Khởi tạo & Nạp cấu hình hệ thống**
2. **Thu nhận khung hình mẫu (Webcam / Ảnh thử nghiệm)**
3. **Phát hiện khuôn mặt & Đếm số lượng người (MediaPipe Face Mesh)**
4. **Trích xuất 6 mốc giải phẫu cốt lõi 2D**
5. **[TRỌNG TÂM] Giải mã thuật toán tính góc đầu (Infographic + solvePnP SQPNP + Chiếu trục 3D)**
6. **Demo tương tác Webcam thời gian thực (Real-time Demo - Nhấn 'q' để dừng)**


---
## ⚙️ Bước 1: Khởi Tạo Môi Trường & Nạp Cấu Hình
Nạp các thư viện chuẩn (`OpenCV`, `MediaPipe`, `NumPy`, `Matplotlib`) và file cấu hình `config.yaml` của dự án.


In [ ]:
# BƯỚC 1: IMPORT THƯ VIỆN & CẤU HÌNH
import sys
import os
import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt

# Đảm bảo đường dẫn nạp được module từ src/
sys.path.insert(0, os.path.abspath("."))
from src.config import get_config

cfg = get_config()
print("✅ Nạp cấu hình thành công:")
print(f"   • Độ phân giải Camera: {cfg.camera.width}x{cfg.camera.height}")
print(f"   • Ngưỡng quay trái/phải (Yaw): {cfg.head_pose.yaw_side_threshold}°")
print(f"   • Ngưỡng quay hẳn ra sau: {cfg.head_pose.yaw_backward_threshold}°")
print(f"   • Ngưỡng cúi/ngửa mặt (Pitch): {cfg.head_pose.pitch_threshold}°")


---
## 📷 Bước 2: Thu Nhận Khung Hình Mẫu (Frame Ingestion)
Chụp 1 khung hình từ Webcam để làm mẫu thử nghiệm cho các bước tiếp theo.  
*(Nếu webcam bận hoặc không có camera, code tự tạo một ảnh chân dung nhân tạo để bạn vẫn chạy demo mượt mà).*


In [ ]:
# BƯỚC 2: BẮT 1 KHUNG HÌNH TỪ WEBCAM (HOẶC SINH ẢNH MẪU)
def get_sample_frame():
    cap = cv2.VideoCapture(cfg.camera.device_id)
    frame = None
    if cap.isOpened():
        # Bỏ qua 5 frame đầu để webcam tự cân bằng sáng
        for _ in range(5):
            ret, tmp = cap.read()
            if ret:
                frame = tmp
        cap.release()

    if frame is None:
        print("⚠️ Không mở được webcam vật lý, tạo ảnh nhân tạo mô phỏng...")
        frame = np.full((480, 640, 3), 45, dtype=np.uint8)
        # Vẽ khuôn mặt giả lập
        cv2.ellipse(frame, (320, 240), (120, 160), 0, 0, 360, (200, 180, 160), -1)
        cv2.circle(frame, (270, 200), 12, (50, 50, 50), -1)  # Mắt trái
        cv2.circle(frame, (370, 200), 12, (50, 50, 50), -1)  # Mắt phải
        cv2.circle(frame, (320, 245), 8, (120, 100, 90), -1) # Mũi
        cv2.ellipse(frame, (320, 310), (45, 15), 0, 0, 180, (80, 60, 180), -1) # Miệng
    else:
        frame = cv2.flip(frame, 1) # Lật gương tự nhiên
        print("✅ Đã chụp thành công 1 frame thật từ Webcam!")

    return frame

sample_bgr = get_sample_frame()
sample_rgb = cv2.cvtColor(sample_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(7, 4.5))
plt.imshow(sample_rgb)
plt.title(f"Khung hình mẫu ({sample_bgr.shape[1]}x{sample_bgr.shape[0]} px)", fontsize=11, fontweight='bold')
plt.axis("off")
plt.show()


---
## 🕸️ Bước 3: Phát Hiện Khuôn Mặt & Đếm Số Lượng Người
* **MediaPipe Face Mesh:** Mô hình học sâu nhận diện khuôn mặt và tạo lưới 468 điểm mốc không gian 3D.
* **Quy tắc đếm người (`num_faces`):**
  * `0 người`: Thí sinh mất mặt khỏi màn hình hoặc che camera.
  * `1 người`: Trạng thái phòng thi hợp lệ (tiến hành đo góc đầu).
  * `> 1 người`: Có người thứ hai xuất hiện trong phòng thi (gian lận nghiêm trọng).


In [ ]:
# BƯỚC 3: CHẠY FACEMESH & ĐẾM SỐ NGƯỜI
mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

face_mesh = mp_face_mesh.FaceMesh(
    max_num_faces=4,
    refine_landmarks=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

results = face_mesh.process(sample_rgb)
num_faces = len(results.multi_face_landmarks) if results.multi_face_landmarks else 0
print(f"📊 Số lượng khuôn mặt phát hiện được: {num_faces} người")

annotated_mesh = sample_rgb.copy()
if results.multi_face_landmarks:
    for face_landmarks in results.multi_face_landmarks:
        mp_drawing.draw_landmarks(
            image=annotated_mesh,
            landmark_list=face_landmarks,
            connections=mp_face_mesh.FACEMESH_TESSELATION,
            landmark_drawing_spec=None,
            connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_tesselation_style()
        )

plt.figure(figsize=(7, 4.5))
plt.imshow(annotated_mesh)
plt.title(f"MediaPipe Face Mesh (468 mốc) - Phát hiện: {num_faces} người", fontsize=11, fontweight='bold', color='navy')
plt.axis("off")
plt.show()


---
## 📍 Bước 4: Trích Xuất 6 Mốc Giải Phẫu Cốt Lõi (Keypoints)
Để tính góc đầu nhanh và chính xác nhất, ta không cần dùng cả 468 điểm mà chỉ trích xuất đúng **6 mốc giải phẫu bất biến**:

| Điểm mốc | Index MediaPipe | Tên giải phẫu |
| :--- | :---: | :--- |
| **Đỉnh mũi** | `1` | Gốc tọa độ, điểm nhô cao nhất |
| **Đáy cằm** | `199` | Định vị trục dọc cằm (Pitch) |
| **Khóe mắt ngoài trái** | `33` | Định vị trục ngang mắt trái (Yaw) |
| **Khóe mắt ngoài phải** | `263` | Định vị trục ngang mắt phải (Yaw) |
| **Khóe miệng ngoài trái** | `61` | Cố định góc miệng bên trái |
| **Khóe miệng ngoài phải** | `291` | Cố định góc miệng bên phải |


In [ ]:
# BƯỚC 4: LẤY TỌA ĐỘ PIXEL 2D CỦA 6 MỐC TRỌNG YẾU
FACE_KEYPOINTS = [1, 199, 33, 263, 61, 291]
h, w, _ = sample_bgr.shape
pts_2d = []

annotated_6pts = sample_rgb.copy()

if results.multi_face_landmarks:
    landmarks = results.multi_face_landmarks[0].landmark
    for idx in FACE_KEYPOINTS:
        px = int(landmarks[idx].x * w)
        py = int(landmarks[idx].y * h)
        pts_2d.append([px, py])
        # Vẽ điểm tròn đỏ và nhãn mốc
        cv2.circle(annotated_6pts, (px, py), 6, (255, 0, 0), -1)
        cv2.circle(annotated_6pts, (px, py), 8, (255, 255, 255), 2)
        cv2.putText(annotated_6pts, str(idx), (px + 10, py - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 2)

pts_2d = np.array(pts_2d, dtype=np.float64)

plt.figure(figsize=(7, 4.5))
plt.imshow(annotated_6pts)
plt.title("6 Điểm Mốc Giải Phẫu Cốt Lõi Được Trích Xuất", fontsize=11, fontweight='bold')
plt.axis("off")
plt.show()

print("Tọa độ 2D (Pixel) trích xuất được:")
for i, idx in enumerate(FACE_KEYPOINTS):
    print(f"  • Mốc {idx:<3}: ({pts_2d[i][0]:.1f}, {pts_2d[i][1]:.1f}) px")


---
## 🧠 Bước 5: Giải Mã Thuật Toán Tính Góc Đầu (Làm Sao Tính Ra Góc Được?)

### ❓ Câu hỏi lớn: *Camera chỉ chụp được ảnh phẳng 2D, làm sao máy tính biết được đầu đang quay bao nhiêu độ trong không gian 3D?*

### 💡 Câu trả lời rất trực quan qua 4 giai đoạn:
1. **Giai đoạn 1 (Có sẵn cái 'khuôn mặt mẫu 3D'):** Các nhà khoa học đo kích thước trung bình khuôn mặt người (tính bằng milimet mm): Mũi cách cằm bao nhiêu, hai mắt cách nhau bao nhiêu $\to$ Ta có 1 mô hình 3D chuẩn cố định (`MODEL_POINTS_3D`).
2. **Giai đoạn 2 (Ảnh 2D bị méo khi quay đầu):** Khi thí sinh ngồi thẳng, mắt trái và mắt phải cách đều mũi. Nhưng khi thí sinh quay đầu sang trái, mắt trái bị co lại gần mũi, mắt phải dạt ra xa $\to$ Ta có 6 tọa độ 2D trên ảnh camera (`pts_2d`).
3. **Giai đoạn 3 (Giải thuật `solvePnP` - Khớp hình 3D vào ảnh 2D):**  
   Thuật toán đặt câu hỏi hình học:  
   > *"Cần phải **XOAY** cái khuôn mặt mẫu 3D một góc bao nhiêu, và **DỊCH CHUYỂN** đi đâu, để khi chiếu tia sáng qua ống kính camera thì các điểm 3D khớp khít khịt lên các điểm 2D trên bức ảnh?"*  
   Hàm `cv2.solvePnP` (dùng giải thuật hiện đại `SOLVEPNP_SQPNP`) giải bài toán này trong chưa đầy $0.5\text{ ms}$, tìm ra Ma trận xoay $R$ ($3\times3$).
4. **Giai đoạn 4 (Phân rã RQ - Đổi ma trận thành góc Độ °):**  
   Ma trận $R$ là một khối 9 con số toán học trừu tượng. Hàm `cv2.RQDecomp3x3` làm nhiệm vụ "rã" ma trận này thành 3 góc Euler quen thuộc:
   * **Pitch (Trục X):** Góc cúi xuống (dương) hoặc ngửa lên (âm).
   * **Yaw (Trục Y):** Góc quay sang trái (dương) hoặc quay sang phải (âm).
   * **Roll (Trục Z):** Góc nghiêng đầu sang vai.
   
> **⚠️ Điểm sáng bảo vệ với thầy:** Hàm `cv2.RQDecomp3x3` của OpenCV **đã trả về kết quả trực tiếp theo độ (°)**. Code của ProctorAI đã **loại bỏ hoàn toàn lỗi nhân sai `* 360`** vốn tồn tại trong nhiều bài hướng dẫn trên mạng!

---
### 📊 Sơ đồ Infographic 4 Giai Đoạn Tính Góc:


In [ ]:
# BƯỚC 5.1: VẼ INFOGRAPHIC TRỰC QUAN HÓA BẢN CHẤT TÍNH GÓC ĐẦU
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8), facecolor='#f8f9fa')

# 1. Mô hình 3D chuẩn
ax1 = axes[0]
ax1.set_facecolor('#ffffff')
ax1.set_title("1. Mô Hình 3D Chuẩn (mm)
(Anthropometric Model)", fontsize=10.5, fontweight='bold', color='#1a365d')
pts_3d_x = [0, 0, -65, 65, -40, 40]
pts_3d_y = [0, -110, 50, 50, -50, -50]
labels = ["Mũi (0,0,0)", "Cằm", "Mắt T", "Mắt P", "Miệng T", "Miệng P"]
ax1.scatter(pts_3d_x, pts_3d_y, color='#e53e3e', s=70, zorder=3)
for i, txt in enumerate(labels):
    ax1.annotate(txt, (pts_3d_x[i]+3, pts_3d_y[i]+3), fontsize=8.5)
ax1.plot([pts_3d_x[2], pts_3d_x[3], pts_3d_x[5], pts_3d_x[1], pts_3d_x[4], pts_3d_x[2]], 
         [pts_3d_y[2], pts_3d_y[3], pts_3d_y[5], pts_3d_y[1], pts_3d_y[4], pts_3d_y[2]], 
         '--', color='#a0aec0', alpha=0.7)
ax1.set_xlim(-100, 100); ax1.set_ylim(-140, 80); ax1.axis('off')
ax1.text(0, -135, "Tọa độ 3D thực nghiệm\ncố định của đầu người", ha='center', fontsize=8.5, color='#4a5568', style='italic')

# 2. Ảnh 2D Camera
ax2 = axes[1]
ax2.set_facecolor('#ffffff')
ax2.set_title("2. Ảnh 2D Camera (Pixel)
(MediaPipe Keypoints)", fontsize=10.5, fontweight='bold', color='#1a365d')
rect = plt.Rectangle((-80, -100), 160, 150, fill=True, facecolor='#edf2f7', edgecolor='#4a5568', lw=1.5)
ax2.add_patch(rect)
pts_2d_x = [-10, -5, -60, 45, -35, 25]
pts_2d_y = [5, -60, 30, 25, -25, -28]
ax2.scatter(pts_2d_x, pts_2d_y, color='#3182ce', s=70, zorder=3)
ax2.plot([pts_2d_x[2], pts_2d_x[3], pts_2d_x[5], pts_2d_x[1], pts_2d_x[4], pts_2d_x[2]], 
         [pts_2d_y[2], pts_2d_y[3], pts_2d_y[5], pts_2d_y[1], pts_2d_y[4], pts_2d_y[2]], 
         '--', color='#3182ce', alpha=0.5)
ax2.set_xlim(-100, 100); ax2.set_ylim(-140, 80); ax2.axis('off')
ax2.text(0, -135, "Khoảng cách các mốc bị méo\nkhi thí sinh quay đầu", ha='center', fontsize=8.5, color='#4a5568', style='italic')

# 3. Thuật toán solvePnP
ax3 = axes[2]
ax3.set_facecolor('#ffffff')
ax3.set_title("3. solvePnP (SQPNP)
(Khớp Mô Hình 3D -> 2D)", fontsize=10.5, fontweight='bold', color='#1a365d')
ax3.plot([0, -70], [60, -80], color='#d69e2e', lw=1.5, ls=':')
ax3.plot([0, 70], [60, -80], color='#d69e2e', lw=1.5, ls=':')
ax3.scatter([0], [60], color='#d69e2e', s=100, marker='v')
box = plt.Rectangle((-65, -50), 130, 60, fill=True, facecolor='#fefcbf', edgecolor='#d69e2e', lw=1.5)
ax3.add_patch(box)
ax3.text(0, -20, "Tìm Vector Xoay (R)\nvà Tịnh Tiến (T)\nsao cho: 3D -> 2D", ha='center', va='center', fontsize=9, fontweight='bold', color='#744210')
ax3.set_xlim(-100, 100); ax3.set_ylim(-140, 80); ax3.axis('off')
ax3.text(0, -135, "Hỏi: 'Xoay đầu góc nào\nđể điểm 3D khớp ảnh 2D?'", ha='center', fontsize=8.5, color='#4a5568', style='italic')

# 4. Tách góc Euler
ax4 = axes[3]
ax4.set_facecolor('#ffffff')
ax4.set_title("4. Tách Góc Euler (Độ °)
(cv2.RQDecomp3x3)", fontsize=10.5, fontweight='bold', color='#1a365d')
colors = ['#e53e3e', '#38a169', '#3182ce']
angles_info = [
    ("Pitch (X): Cúi/Ngửa", "+12.4°", 0.6),
    ("Yaw (Y): Trái/Phải", "-28.7°", 0.2),
    ("Roll (Z): Nghiêng vai", "+3.1°", -0.2)
]
for i, (name, val, y_pos) in enumerate(angles_info):
    card = plt.Rectangle((-85, y_pos*60 - 15), 170, 30, fill=True, facecolor='#edf2f7', edgecolor=colors[i], lw=2)
    ax4.add_patch(card)
    ax4.text(-75, y_pos*60, name, va='center', fontsize=8, fontweight='bold', color='#2d3748')
    ax4.text(75, y_pos*60, val, va='center', ha='right', fontsize=9, fontweight='bold', color=colors[i])
ax4.set_xlim(-100, 100); ax4.set_ylim(-140, 80); ax4.axis('off')
ax4.text(0, -135, "Góc Euler chuẩn (°)\n(Không nhân thêm 360)", ha='center', fontsize=8.5, color='#4a5568', style='italic')

plt.tight_layout()
plt.show()


---
### 💻 Bước 5.2: Thực Thi Code Tính Góc & Chiếu 3 Trục Tọa Độ Lên Mặt
*Sau khi tính được góc Pitch/Yaw, ta dùng hàm `cv2.projectPoints` để chiếu 3 trục từ đỉnh mũi ra không gian ảnh 2D:*
* 🔴 **Trục X (Đỏ):** Trục quay Pitch (hướng sang ngang).
* 🟢 **Trục Y (Xanh lá):** Trục quay Yaw (hướng thẳng xuống đất).
* 🔵 **Trục Z (Xanh dương):** Vector hướng nhìn (đâm thẳng từ chóp mũi ra ngoài - mũi chỉ đi đâu nghĩa là mắt nhìn về đó!).


In [ ]:
# BƯỚC 5.2: TÍNH GÓC ĐẦU BẰNG SQPNP & VẼ 3 TRỤC TỌA ĐỘ TỪ ĐỈNH MŨI
# 1. Định nghĩa 6 điểm 3D chuẩn trong không gian (mm)
MODEL_POINTS_3D = np.array([
    (0.0, 0.0, 0.0),             # 1: Đỉnh mũi (Gốc tọa độ)
    (0.0, 110.0, 65.0),          # 199: Đáy cằm
    (-65.0, -50.0, 60.0),        # 33: Khóe mắt ngoài trái
    (65.0, -50.0, 60.0),         # 263: Khóe mắt ngoài phải
    (-40.0, 50.0, 40.0),         # 61: Khóe miệng ngoài trái
    (40.0, 50.0, 40.0)           # 291: Khóe miệng ngoài phải
], dtype=np.float64)

# 2. Ma trận camera nội suy K
cam_matrix = np.array([
    [w, 0, w / 2],
    [0, w, h / 2],
    [0, 0, 1]
], dtype=np.float64)
dist_coeffs = np.zeros((4, 1), dtype=np.float64)

# 3. Giải thuật solvePnP (SQPNP) - Khớp 3D sang 2D
success, rot_vec, trans_vec = cv2.solvePnP(
    MODEL_POINTS_3D,
    pts_2d,
    cam_matrix,
    dist_coeffs,
    flags=cv2.SOLVEPNP_SQPNP
)

# 4. Đổi vector xoay sang Ma trận quay R và rã góc Euler
rot_mat, _ = cv2.Rodrigues(rot_vec)
angles, _, _, _, _, _ = cv2.RQDecomp3x3(rot_mat)

pitch = round(float(angles[0]), 1)
yaw = round(float(angles[1]), 1)
roll = round(float(angles[2]), 1)

print(f"📊 Góc Euler đo được: Pitch={pitch:+.1f}° | Yaw={yaw:+.1f}° | Roll={roll:+.1f}°")

# 5. Chiếu 3 trục tọa độ 3D từ đỉnh mũi ra ảnh 2D
axis_length = 80.0
axis_3d = np.array([
    (axis_length, 0, 0),    # Trục X (Đỏ)
    (0, axis_length, 0),    # Trục Y (Xanh lá)
    (0, 0, axis_length)     # Trục Z (Xanh dương - Vector hướng nhìn)
], dtype=np.float64)

axis_2d, _ = cv2.projectPoints(axis_3d, rot_vec, trans_vec, cam_matrix, dist_coeffs)

nose_pt = (int(pts_2d[0][0]), int(pts_2d[0][1]))
x_pt = (int(axis_2d[0][0][0]), int(axis_2d[0][0][1]))
y_pt = (int(axis_2d[1][0][0]), int(axis_2d[1][0][1]))
z_pt = (int(axis_2d[2][0][0]), int(axis_2d[2][0][1]))

annotated_axes = sample_rgb.copy()
cv2.line(annotated_axes, nose_pt, x_pt, (255, 0, 0), 3)   # Trục X (Đỏ)
cv2.line(annotated_axes, nose_pt, y_pt, (0, 255, 0), 3)   # Trục Y (Xanh lá)
cv2.line(annotated_axes, nose_pt, z_pt, (0, 100, 255), 4) # Trục Z (Xanh dương - Mũi chỉ hướng nào đầu nhìn hướng đó)
cv2.circle(annotated_axes, nose_pt, 5, (255, 255, 0), -1)

# Đánh giá vi phạm theo ngưỡng trong config.yaml
status = "BÌNH THƯỜNG - HỢP LỆ"
if abs(yaw) > cfg.head_pose.yaw_backward_threshold:
    status = "CẢNH BÁO: QUAY HẲN RA SAU"
elif yaw > cfg.head_pose.yaw_side_threshold:
    status = "CHÚ Ý: QUAY SANG TRÁI"
elif yaw < -cfg.head_pose.yaw_side_threshold:
    status = "CHÚ Ý: QUAY SANG PHẢI"
elif pitch > cfg.head_pose.pitch_threshold:
    status = "CHÚ Ý: CÚI ĐẦU XUỐNG GẦM BÀN"
elif pitch < -cfg.head_pose.pitch_threshold:
    status = "CHÚ Ý: NGỬA MẶT LÊN TRẦN"

plt.figure(figsize=(7.5, 5))
plt.imshow(annotated_axes)
plt.title(f"Chiếu Trục Tọa Độ 3D (Pose Axes)\nPitch: {pitch}° | Yaw: {yaw}° -> {status}", 
          fontsize=11, fontweight='bold', color='darkred' if "CẢNH BÁO" in status else 'darkgreen')
plt.axis("off")
plt.show()


---
## 🔴 Bước 6: Demo Tương Tác Webcam Thời Gian Thực (Interactive Real-Time)
Chạy ô code bên dưới để mở cửa sổ Webcam tương tác trực tiếp với thầy cô:
* Bạn quay đầu sang trái, phải, cúi mặt, ngửa mặt hoặc ngoái hẳn ra sau để xem góc nhảy và cảnh báo cập nhật theo thời gian thực.
* **Nhấn phím `q` trên cửa sổ OpenCV để dừng lại.**


In [ ]:
# BƯỚC 6: VÒNG LẶP WEBCAM THỜI GIAN THỰC (NHẤN 'q' ĐỂ THOÁT)
from src.head_pose import HeadPoseEstimator

cap = cv2.VideoCapture(cfg.camera.device_id)
if not cap.isOpened():
    print("❌ Không mở được Webcam. Hãy đảm bảo webcam không bị ứng dụng khác chiếm giữ.")
else:
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, cfg.camera.width)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, cfg.camera.height)
    estimator = HeadPoseEstimator(cfg.camera.width, cfg.camera.height)
    
    print("🚀 Đang khởi động cửa sổ Webcam Demo... Nhấn phím 'q' trên cửa sổ để thoát.")
    
    while True:
        ret, frame = cap.read()
        if not ret or frame is None:
            break
            
        frame = cv2.flip(frame, 1) # Lật gương
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Gọi thuật toán ước lượng góc đầu
        pitch, yaw, violation = estimator.process(frame_rgb)
        
        # Vẽ HUD thông báo
        color = (0, 255, 0)
        status_str = "BINH THUONG - HOP LE"
        if violation:
            color = (0, 0, 255) if "sau" in violation.lower() or "người" in violation.lower() else (0, 255, 255)
            status_str = f"CHU Y: {violation.upper()}"
            
        cv2.rectangle(frame, (10, 10), (630, 85), (20, 20, 20), -1)
        cv2.rectangle(frame, (10, 10), (630, 85), color, 2)
        cv2.putText(frame, f"STATUS: {status_str}", (25, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)
        cv2.putText(frame, f"Pitch: {pitch:+.1f} deg | Yaw: {yaw:+.1f} deg", (25, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
        cv2.imshow("ProctorAI - Head Pose Demo (Nhan 'q' de thoat)", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
            
    cap.release()
    cv2.destroyAllWindows()
    estimator.close()
    print("👋 Đã đóng camera demo thành công!")
